# Lesson 9B: Association Rules - Practical

## Grocery Store Analysis

**Case Study**: A grocery chain wants to optimize product placement and create targeted promotions. Which products should be placed together?

**Goal**: Find actionable product associations for cross-selling and store layout.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

np.random.seed(42)
print("✅ Loaded!")

## Step 1: Simulate Realistic Grocery Data

We'll create a dataset mimicking real grocery shopping patterns.

In [ ]:
# Realistic grocery transactions
# Common patterns: breakfast items, dinner items, snacks, healthy foods
transactions = []

# Generate 200 transactions with realistic patterns
for _ in range(50):
    transactions.append(['milk', 'bread', 'eggs'])  # Breakfast
for _ in range(40):
    transactions.append(['milk', 'bread', 'butter', 'cheese'])  # Dairy lovers
for _ in range(30):
    transactions.append(['bread', 'eggs', 'butter'])  # Baking
for _ in range(25):
    transactions.append(['milk', 'eggs'])  # Quick breakfast
for _ in range(20):
    transactions.append(['bread', 'butter', 'cheese'])  # Sandwich
for _ in range(15):
    transactions.append(['beer', 'chips', 'salsa'])  # Party
for _ in range(10):
    transactions.append(['yogurt', 'fruit', 'granola'])  # Healthy
for _ in range(10):
    transactions.append(['pasta', 'sauce', 'cheese'])  # Italian dinner

print(f"Total transactions: {len(transactions)}")
print(f"\nSample transactions:")
for t in transactions[:5]:
    print(f"  {t}")
print("✅ Data generated!")

## Step 2: Apply Apriori Algorithm

In [ ]:
# Encode transactions
te = TransactionEncoder()
te_ary = te.fit_transform(transactions)
df = pd.DataFrame(te_ary, columns=te.columns_)

print(f"One-hot encoded shape: {df.shape}")
print(f"Unique items: {df.shape[1]}")
print(f"\nItem frequencies:")
print(df.sum().sort_values(ascending=False))

# Find frequent itemsets
frequent_itemsets = apriori(df, min_support=0.1, use_colnames=True)
print(f"\nFrequent itemsets found: {len(frequent_itemsets)}")
print(frequent_itemsets.sort_values('support', ascending=False).head(10))
print("✅ Frequent itemsets mined!")

## Step 3: Generate Association Rules

In [ ]:
# Generate rules
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.2)
rules = rules.sort_values('lift', ascending=False)

print(f"Total rules: {len(rules)}")
print(f"\nTop 10 Rules by Lift:")
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10))

# Filter for actionable rules (high confidence + high lift)
actionable = rules[(rules['confidence'] >= 0.6) & (rules['lift'] >= 1.5)]
print(f"\n=== Actionable Rules (confidence ≥ 60%, lift ≥ 1.5) ===")
print(f"Found {len(actionable)} actionable rules:")
for idx, row in actionable.head(10).iterrows():
    print(f"  {set(row['antecedents'])} → {set(row['consequents'])}")
    print(f"    Confidence: {row['confidence']:.1%}, Lift: {row['lift']:.2f}")
print("✅ Rules generated!")

## Step 4: Parameter Tuning

Finding optimal min_support and min_confidence thresholds.

In [ ]:
# Test different support thresholds
support_thresholds = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3]
results = []

for min_sup in support_thresholds:
    freq = apriori(df, min_support=min_sup, use_colnames=True)
    if len(freq) > 0:
        rules = association_rules(freq, metric="lift", min_threshold=1.0)
        results.append({
            'min_support': min_sup,
            'frequent_itemsets': len(freq),
            'total_rules': len(rules),
            'strong_rules': len(rules[rules['lift'] >= 1.5])
        })

results_df = pd.DataFrame(results)
print(results_df)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(results_df['min_support'], results_df['frequent_itemsets'], 
       'bo-', linewidth=2, markersize=8, label='Frequent Itemsets')
ax.set_xlabel('Minimum Support', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Frequent Itemsets vs Support', fontweight='bold', fontsize=13)
ax.grid(True, alpha=0.3)
ax.legend()

ax = axes[1]
ax.plot(results_df['min_support'], results_df['total_rules'], 
       'go-', linewidth=2, markersize=8, label='All Rules')
ax.plot(results_df['min_support'], results_df['strong_rules'], 
       'ro-', linewidth=2, markersize=8, label='Strong Rules (lift ≥ 1.5)')
ax.set_xlabel('Minimum Support', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Rules vs Support', fontweight='bold', fontsize=13)
ax.grid(True, alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()
print("✅ Parameter tuning complete!")

## Step 5: Visualize Association Rules

In [ ]:
# Use optimal parameters
frequent_itemsets = apriori(df, min_support=0.1, use_colnames=True)
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.2)

# Comprehensive visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. Support vs Confidence (size = lift)
ax = axes[0, 0]
scatter = ax.scatter(rules['support'], rules['confidence'], 
                    s=rules['lift']*50, alpha=0.6, c=rules['lift'], 
                    cmap='plasma', edgecolors='black', linewidths=0.5)
ax.set_xlabel('Support', fontsize=11)
ax.set_ylabel('Confidence', fontsize=11)
ax.set_title('Support vs Confidence (size & color = lift)', fontweight='bold')
ax.grid(True, alpha=0.3)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Lift')

# 2. Lift distribution
ax = axes[0, 1]
ax.hist(rules['lift'], bins=30, alpha=0.7, color='steelblue', edgecolor='black')
ax.axvline(1.0, color='red', linestyle='--', linewidth=2, label='Independent')
ax.axvline(1.5, color='orange', linestyle='--', linewidth=2, label='Strong')
ax.set_xlabel('Lift', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.set_title('Lift Distribution', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# 3. Confidence distribution
ax = axes[1, 0]
ax.hist(rules['confidence'], bins=30, alpha=0.7, color='seagreen', edgecolor='black')
ax.axvline(0.7, color='red', linestyle='--', linewidth=2, label='70% threshold')
ax.set_xlabel('Confidence', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.set_title('Confidence Distribution', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# 4. Top rules by lift
ax = axes[1, 1]
top_rules = rules.nlargest(10, 'lift')
rule_labels = [f"{list(r['antecedents'])[0][:4]}→{list(r['consequents'])[0][:4]}" 
              for _, r in top_rules.iterrows()]
ax.barh(range(len(top_rules)), top_rules['lift'], color='coral', edgecolor='black')
ax.set_yticks(range(len(top_rules)))
ax.set_yticklabels(rule_labels, fontsize=9)
ax.set_xlabel('Lift', fontsize=11)
ax.set_title('Top 10 Rules by Lift', fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()
print("✅ Visualization complete!")

## Step 6: Actionable Business Insights

In [ ]:
# Extract top actionable insights
print("=== ACTIONABLE BUSINESS INSIGHTS ===\n")

# Get strongest rules
strong_rules = rules[(rules['confidence'] >= 0.6) & (rules['lift'] >= 1.5)].nlargest(5, 'lift')

for idx, (i, rule) in enumerate(strong_rules.iterrows(), 1):
    ant = list(rule['antecedents'])[0]
    cons = list(rule['consequents'])[0]
    
    print(f"{idx}. RECOMMENDATION: Place {cons} near {ant}")
    print(f"   → {rule['confidence']:.0%} of customers who buy {ant} also buy {cons}")
    print(f"   → {rule['lift']:.1f}x more likely than random")
    print(f"   → Affects {rule['support']:.0%} of all transactions")
    print(f"   → ACTION: Cross-sell promotion or shelf placement")
    print()

print("=== PRODUCTION DEPLOYMENT ===")
print("1. Update store layout based on top rules")
print("2. Create 'Frequently Bought Together' promotions")
print("3. Train recommendation system for online orders")
print("4. A/B test new layouts and measure sales lift")
print("5. Re-run analysis quarterly to detect changing patterns")
print("\n✅ Market basket analysis complete!")